# UniProt RAG — query & summarize

Prerequisites: `python src/ingest_uniprot.py` then `python src/build_uniprot_index.py`. EDA: `uniprot_eda.ipynb`.

In [ ]:
# !pip install chromadb sentence-transformers transformers huggingface_hub accelerate

In [ ]:
from pathlib import Path

import chromadb
import pandas as pd
from sentence_transformers import SentenceTransformer

In [ ]:
project_root = Path("..").resolve()
data_processed = project_root / "data/processed"
chroma_dir = project_root / "data/chroma_db"

UNIPROT_PARQUET = data_processed / "uniprot_rag.parquet"
UNIPROT_COLLECTION = "protein_chunks"
EMBED_MODEL = "BAAI/bge-small-en-v1.5"

## Search

In [ ]:
query = input("Enter your UniProt query: ").strip()
if not query:
    raise ValueError("Query cannot be empty.")

TOP_K_CHUNKS = 20
TOP_K_PROTEINS = 5

model = SentenceTransformer(EMBED_MODEL)
client = chromadb.PersistentClient(path=str(chroma_dir))
collection = client.get_collection(UNIPROT_COLLECTION)
print(f"Collection {UNIPROT_COLLECTION}: {collection.count()} chunks")

results = collection.query(
    query_embeddings=[model.encode(query).tolist()],
    n_results=TOP_K_CHUNKS,
    include=["documents", "metadatas", "distances"],
)

best_by_entry_name = {}
chunks_by_entry = {}
for doc_text, meta, dist in zip(
    results["documents"][0], results["metadatas"][0], results["distances"][0],
):
    entry_name = (meta or {}).get("entry_name")
    hit = {"text": doc_text, "metadata": meta, "distance": dist}
    chunks_by_entry.setdefault(entry_name, []).append(hit)
    if entry_name not in best_by_entry_name or dist < best_by_entry_name[entry_name]["distance"]:
        best_by_entry_name[entry_name] = hit

ranked = sorted(best_by_entry_name.values(), key=lambda x: x["distance"])[:TOP_K_PROTEINS]
print(f"Query: {query}\n")
for rank, hit in enumerate(ranked, 1):
    m = hit["metadata"] or {}
    n_chunks = len(chunks_by_entry.get(m.get("entry_name"), []))
    print(
        f"{rank}. entry_name={m.get('entry_name')} accession={m.get('accession')} "
        f"gene={m.get('gene')} best_chunk={m.get('chunk_type')} dist={hit['distance']:.4f} "
        f"({n_chunks} retrieved chunk(s))"
    )
    print(hit["text"][:500] + ("..." if len(hit["text"]) > 500 else ""))
    print()

selected_accessions = [(h["metadata"] or {}).get("accession") for h in ranked]
df = pd.read_parquet(UNIPROT_PARQUET)
df = df[df["Entry"].isin(selected_accessions)].copy()
print(f"Selected {len(df)} protein record(s) for context")

In [ ]:
UNIPROT_RECORD_FIELDS = [
    "Entry", "Entry Name", "Gene Names", "Protein names", "Length",
    "Function [CC]", "Involvement in disease", "Subcellular location [CC]",
    "Domain [FT]", "Region", "Zinc finger", "Domain [CC]",
    "Natural variant", "Polymorphism", "Tissue specificity",
    "Developmental stage", "Interacts with",
]


def format_protein_record(row: pd.Series) -> str:
    return "\n".join(
        f"{col}: {row[col]}"
        for col in UNIPROT_RECORD_FIELDS
        if col in row.index and pd.notna(row[col])
    )


def build_rag_context(query: str, ranked: list, chunks_by_entry: dict, records_df: pd.DataFrame) -> str:
    sections = []
    for rank, hit in enumerate(ranked, 1):
        meta = hit.get("metadata") or {}
        entry_name = str(meta.get("entry_name"))
        accession = str(meta.get("accession"))
        row = records_df.loc[records_df["Entry"] == accession]
        full_record = format_protein_record(row.iloc[0]) if len(row) else "(record not found)"

        entry_chunks = sorted(chunks_by_entry.get(entry_name, [hit]), key=lambda x: x["distance"])
        chunk_parts = []
        for i, chunk in enumerate(entry_chunks, 1):
            cm = chunk.get("metadata") or {}
            chunk_parts.append(
                f"Chunk {i} ({cm.get('chunk_type')}, dist={chunk['distance']:.4f}):\n{chunk['text']}"
            )

        sections.append(
            f"### Result {rank} — entry_name={entry_name}, accession={accession}, gene={meta.get('gene')}\n\n"
            f"Retrieved chunks:\n" + "\n\n".join(chunk_parts)
            + "\n\nFull UniProt record:\n" + full_record
        )
    return f"User query: {query}\n\n" + "\n\n---\n\n".join(sections)


rag_context = build_rag_context(query, ranked, chunks_by_entry, df)
print(rag_context[:4000] + ("..." if len(rag_context) > 4000 else ""))

## Summarize

In [ ]:
import os

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"
LOAD_LLM = True
MAX_NEW_TOKENS = 1024

SYSTEM_PROMPT = (
    "You are a protein biology assistant. Summarize UniProt protein search results "
    "for the user's query using ONLY the retrieved chunks and full records provided. "
    "Be concise, accurate, and structured. For each relevant protein, mention entry name, "
    "accession, gene, function, disease involvement, and tissue expression when available. "
    "If information is missing, say so. Do not invent facts."
)


def summarize_rag(query: str, context: str, model, tokenizer, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Query: {query}\n\n"
                f"Retrieved evidence:\n\n{context}\n\n"
                "Write a short summary answering the query from this evidence."
            ),
        },
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = output_ids[0][len(inputs.input_ids[0]):]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


if LOAD_LLM:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        print("Warning: no GPU — 7B on CPU is slow but workable (~14GB RAM).")

    hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")
    tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL, token=hf_token)
    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        token=hf_token,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto" if device == "cuda" else None,
    )
    if device == "cpu":
        model = model.to(device)

    summary = summarize_rag(query, rag_context, model, tokenizer)
    print("=" * 60)
    print(f"Query: {query}\n")
    print(summary)
else:
    print("LOAD_LLM is False — set True to run summarization.")